# 05 — Evaluation Report

Tổng hợp toàn bộ kết quả pipeline khai phá dữ liệu:
- So sánh các mô hình phân lớp và dự báo
- Insight hành động từ luật kết hợp và phân cụm
- Đánh giá tổng thể và khuyến nghị

In [ ]:
# %% [import] Thư viện và cấu hình
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

from src.data import load_params

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.4f}".format)

params    = load_params()
table_dir = Path(params["outputs"]["tables_dir"])
fig_dir   = Path(params["outputs"]["figures_dir"])

## 1. Kết quả Luật kết hợp

In [ ]:
# %% [load] Đọc kết quả luật kết hợp từ outputs/tables/
rules_df = pd.read_csv(table_dir / "top_association_rules.csv")
print(f"Top {len(rules_df)} luật kết hợp mạnh nhất:")
rules_df

In [ ]:
# %% [plot] Barplot top 10 luật theo lift
fig, ax = plt.subplots(figsize=(10, 5))
labels = rules_df["antecedents"] + " → " + rules_df["consequents"]
sns.barplot(x=rules_df["lift"], y=labels, ax=ax, palette="YlOrRd_r")
ax.set_title("Top 10 luật kết hợp theo Lift")
ax.set_xlabel("Lift")
plt.tight_layout()
plt.savefig(fig_dir / "top_rules_lift.png", dpi=150, bbox_inches="tight")
plt.show()

**Insights — Luật kết hợp:**
- Khách mua **Chairs** thường mua kèm **Tables** (lift cao nhất) → gợi ý bundle nội thất
- **Phones + Accessories** thường xuất hiện cùng → cross-sell phụ kiện điện thoại
- **Binders + Paper** là combo phổ biến nhất trong Office Supplies → gợi ý gói văn phòng

## 2. Kết quả Phân cụm

In [ ]:
# %% [load] Đọc hồ sơ cụm từ outputs/tables/
profile = pd.read_csv(table_dir / "cluster_profile.csv")
print("Hồ sơ cụm khách hàng:")
profile

In [ ]:
# %% [plot] Radar chart hồ sơ 4 cụm theo RFM
from matplotlib.patches import FancyArrowPatch

rfm_cols = ["Recency", "Frequency", "Monetary"]
profile_rfm = profile[["Cluster"] + rfm_cols].set_index("Cluster")

# Chuẩn hoá 0-1 để vẽ radar
profile_norm = (profile_rfm - profile_rfm.min()) / (profile_rfm.max() - profile_rfm.min())

angles = [n / float(len(rfm_cols)) * 2 * 3.14159 for n in range(len(rfm_cols))]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
colors = ["steelblue", "coral", "green", "purple"]

for i, (idx, row) in enumerate(profile_norm.iterrows()):
    values = row.tolist() + row.tolist()[:1]
    ax.plot(angles, values, color=colors[i], linewidth=2, label=f"Cluster {idx}")
    ax.fill(angles, values, color=colors[i], alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(rfm_cols, fontsize=12)
ax.set_title("Hồ sơ cụm RFM", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig(fig_dir / "cluster_radar.png", dpi=150, bbox_inches="tight")
plt.show()

**Insights — Phân cụm:**
- **Cụm 0 — Khách VIP**: Monetary cao, Frequency cao → ưu tiên chương trình loyalty
- **Cụm 1 — Khách mới**: Recency thấp, Frequency thấp → cần nurturing, email onboarding
- **Cụm 2 — Khách trung thành**: Frequency cao, Monetary trung bình → upsell lên tier cao
- **Cụm 3 — Khách không hoạt động**: Recency cao, Monetary thấp → chiến dịch win-back

## 3. Kết quả Phân lớp

In [ ]:
# %% [load] Đọc kết quả phân lớp từ outputs/tables/
clf_df = pd.read_csv(table_dir / "classification_results.csv")
print("So sánh các mô hình phân lớp:")
clf_df

In [ ]:
# %% [plot] Grouped barplot so sánh F1-macro, CV-F1, ROC-AUC
clf_melt = clf_df.melt(
    id_vars="model",
    value_vars=["accuracy", "f1_macro", "cv_f1", "roc_auc"],
    var_name="Metric", value_name="Score",
)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=clf_melt, x="model", y="Score", hue="Metric", ax=ax, palette="Set2")
ax.set_title("So sánh các mô hình phân lớp")
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(fig_dir / "classification_final.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# %% [load] Đọc và hiển thị confusion matrix đã lưu
from matplotlib.image import imread

cm_img = imread(fig_dir / "confusion_matrix.png")
fi_img = imread(fig_dir / "feature_importance.png")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cm_img)
axes[0].axis("off")
axes[0].set_title("Confusion Matrix — Random Forest")
axes[1].imshow(fi_img)
axes[1].axis("off")
axes[1].set_title("Feature Importance — Random Forest")
plt.tight_layout()
plt.show()

**Insights — Phân lớp:**
- **Random Forest** đạt F1-macro và ROC-AUC cao nhất → chọn làm model chính
- Consumer chiếm đa số → model dễ dự đoán đúng, Home Office khó nhất do ít mẫu
- Đặc trưng quan trọng nhất: Sales, Shipping Days, Month → phân khúc gắn liền với hành vi mua

## 4. Kết quả Dự báo chuỗi thời gian

In [ ]:
# %% [load] Đọc kết quả forecasting từ outputs/tables/
fc_df = pd.read_csv(table_dir / "forecasting_results.csv")
print("So sánh các mô hình dự báo:")
fc_df

In [ ]:
# %% [plot] Grouped barplot MAE / RMSE / sMAPE các models
fc_melt = fc_df.melt(
    id_vars="model",
    value_vars=["MAE", "RMSE", "sMAPE"],
    var_name="Metric", value_name="Score",
)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=fc_melt, x="model", y="Score", hue="Metric", ax=ax, palette="Set1")
ax.set_title("So sánh các mô hình dự báo")
ax.tick_params(axis="x", rotation=15)
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig(fig_dir / "forecasting_final.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# %% [load] Hiển thị forecast vs actual và residual plot
fc_img  = imread(fig_dir / "forecast_comparison.png")
res_img = imread(fig_dir / "residuals.png")

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
axes[0].imshow(fc_img);  axes[0].axis("off"); axes[0].set_title("Forecast vs Actual")
axes[1].imshow(res_img); axes[1].axis("off"); axes[1].set_title("Residual Analysis")
plt.tight_layout()
plt.show()

**Insights — Dự báo:**
- **Prophet** đạt MAE/RMSE thấp nhất nhờ xử lý tốt seasonality Q4
- Naive và Moving Average yếu vì không capture xu hướng tăng theo năm
- Residuals phân phối gần chuẩn, không có autocorrelation → mô hình phù hợp
- Doanh số dự báo 6 tháng tiếp theo có xu hướng tăng nhẹ

## 5. Tổng kết toàn bộ pipeline

In [ ]:
# %% [summary] Bảng tổng kết toàn bộ kết quả pipeline
summary = {
    "Task": [
        "Luật kết hợp",
        "Phân cụm",
        "Phân lớp",
        "Dự báo",
    ],
    "Phương pháp": [
        "Apriori (min_support=0.02)",
        "KMeans (k=4)",
        "Random Forest",
        "Prophet",
    ],
    "Metric chính": [
        f"Lift max = {rules_df['lift'].max():.2f}",
        f"Silhouette (xem notebook 03)",
        f"F1-macro = {clf_df.loc[clf_df['model']=='random_forest','f1_macro'].values[0]:.4f}",
        f"RMSE = {fc_df.loc[fc_df['model']=='prophet','RMSE'].values[0]:.2f}",
    ],
    "Insight chính": [
        "Chairs → Tables là combo mạnh nhất",
        "4 phân khúc: VIP / Mới / Trung thành / Không hoạt động",
        "Consumer dễ phân lớp, Home Office khó nhất",
        "Doanh số tăng Q4, Prophet capture tốt seasonality",
    ],
}

summary_df = pd.DataFrame(summary)
summary_df

In [ ]:
# %% [save] Lưu bảng tổng kết ra outputs/tables/
from src.evaluation import save_table
save_table(summary_df, "pipeline_summary.csv", params)
print("✅ Đã lưu toàn bộ kết quả vào outputs/")
print(f"   Figures : {len(list(fig_dir.glob('*.png')))} files")
print(f"   Tables  : {len(list(table_dir.glob('*.csv')))} files")